# Woche 2 · Python für AI: NumPy, APIs und der erste LLM-Call
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Letzte Woche habt ihr 24 Störungsmeldungen mit Schleifen ausgewertet. Heute geht es um drei Dinge, die jedes AI-System braucht:

1. **Viele Zahlen auf einmal rechnen** mit NumPy, inklusive der Kosinus-Ähnlichkeit, mit der später Embeddings verglichen werden.
2. **Mit Webdiensten sprechen** über HTTP und JSON.
3. **Euer erster eigener Request an ein Sprachmodell** über die KI-Toolbox des KIT, inklusive Token-Verbrauch.

**Lernziele**
- Vektoren und Matrizen in NumPy nutzen, inklusive Broadcasting
- HTTP-Requests absetzen und JSON verarbeiten
- API-Keys sicher über Colab-Secrets verwalten
- Einen LLM-Call absetzen, Antwort und Token-Verbrauch auslesen und den Ressourcenbedarf einschätzen

**Wie letzte Woche:** Kopie in Drive speichern, Zellen von oben nach unten ausführen, `# TODO` ausfüllen, Selbsttests beachten, 10-Minuten-Regel.
Für Teil C braucht ihr euren Toolbox-Schlüssel als Colab-Secret `KIT_TOOLBOX_KEY` (Woche 0, Kapitel 8).

In [ ]:
# Setup: installiert die OpenAI-Bibliothek in fester Version (dauert ca. 10 Sekunden).
# Sie spricht mit allen OpenAI-kompatiblen Diensten, auch mit der KI-Toolbox des KIT.
%pip install -q openai==3.24.0

In [ ]:
import math, time, json
from collections import Counter
from types import SimpleNamespace

import numpy as np
import requests
import matplotlib.pyplot as plt
import openai
from openai import OpenAI

# NumPy, requests und matplotlib sind in Colab vorinstalliert. Wir installieren sie bewusst nicht neu,
# weil ein Neuinstallieren von NumPy in Colab einen Neustart der Laufzeit erzwingt.
print("NumPy", np.__version__, "| requests", requests.__version__, "| openai", openai.__version__)

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

---
# Teil A · NumPy: rechnen mit ganzen Vektoren

## A1 · Arrays statt Listen

Ein NumPy-**Array** speichert viele Zahlen **eines Typs** dicht hintereinander im Speicher. Rechenoperationen wirken **elementweise** auf das ganze Array, ohne dass ihr eine Schleife schreibt.
Zwei Eigenschaften braucht ihr ständig: `shape` (die Form) und `dtype` (der Zahlentyp).

In [ ]:
# Die 24 Stillstandszeiten aus Woche 1, in der Reihenfolge S-001 bis S-024
STILLSTAENDE = [45, 20, 35, 240, 60, 25, 15, 35, 50, 40, 70, 30, 90, 30, 55, 10, 40, 65, 20, 25, 55, 45, 35, 75]

s = np.array(STILLSTAENDE)
print("shape:", s.shape, "| dtype:", s.dtype)
print("In Stunden:", np.round(s / 60, 2)[:6], "...")      # elementweise, ohne Schleife
print("Summe:", s.sum(), "| Mittelwert:", s.mean().round(2))
print("np.std(s)         =", np.std(s).round(3), " ← teilt durch n (wie statistics.pstdev)")
print("np.std(s, ddof=1) =", np.std(s, ddof=1).round(3), " ← teilt durch n−1 (wie statistics.stdev)")

**Boolesche Masken** ersetzen das Filtern mit Schleife aus Woche 1. Ein Vergleich wie `s > 60` liefert ein Array aus `True`/`False`. Damit kann man indizieren: `s[s > 60]`.
Und weil `True` als 1 zählt, ist `np.mean(s > 60)` direkt der **Anteil** der Werte über 60.

### TODO 1 · Lange Stillstände
Berechnet ohne Schleife:
- `lang`: alle Stillstände **über 60 Minuten**
- `anteil_lang`: den Anteil dieser Stillstände an allen Meldungen (eine Zahl zwischen 0 und 1)

In [ ]:
# TODO 1
lang = None
anteil_lang = None

print(lang, anteil_lang)

In [ ]:
check("TODO 1a lang", lambda: sorted(lang.tolist()) == [65, 70, 75, 90, 240])
check("TODO 1b anteil_lang", lambda: math.isclose(anteil_lang, 5 / 24))

## A2 · Wie viel schneller ist NumPy?

Das ist die Einstiegsfrage von heute. Das **Skalarprodukt** zweier Vektoren ist $a \cdot b = \sum_i a_i b_i$.

### TODO 2 · Skalarprodukt in reinem Python
Schreibt `skalarprodukt_python(a, b)` mit einer Schleife (Akkumulator-Muster aus Woche 1). Tipp: `zip(a, b)` liefert Paare.

In [ ]:
def skalarprodukt_python(a, b):
    # TODO 2
    pass

In [ ]:
check("TODO 2 kleines Beispiel", lambda: skalarprodukt_python([1, 2, 3], [4, 5, 6]) == 32)

Jetzt das Rennen. `N` sind 10 Millionen Werte. Falls Colab wegen Speicher meckert, setzt `N = 1_000_000`.

In [ ]:
N = 10_000_000
rng = np.random.default_rng(0)
a = rng.random(N)
b = rng.random(N)
a_liste, b_liste = a.tolist(), b.tolist()       # dieselben Zahlen als normale Python-Listen

if skalarprodukt_python([1], [1]) is None:
    print("⏳ Erst TODO 2 lösen.")
else:
    t0 = time.perf_counter(); ergebnis_python = skalarprodukt_python(a_liste, b_liste); t_python = time.perf_counter() - t0
    t0 = time.perf_counter(); ergebnis_numpy = a @ b;                                   t_numpy = time.perf_counter() - t0
    print(f"Python-Schleife: {t_python:.3f} s   ergebnis = {ergebnis_python:.6f}")
    print(f"NumPy  a @ b:    {t_numpy:.4f} s   ergebnis = {ergebnis_numpy:.6f}")
    print(f"NumPy ist hier etwa {t_python / t_numpy:.0f}-mal schneller.")
    print("Exakt gleich?", ergebnis_python == ergebnis_numpy, "| fast gleich?", math.isclose(ergebnis_python, ergebnis_numpy, rel_tol=1e-9))

> **Zum Nachdenken:** Warum sind die beiden Ergebnisse in den letzten Stellen verschieden? (Hinweis: Gleitkomma-Addition ist nicht assoziativ, und NumPy summiert in anderer Reihenfolge. Deshalb vergleichen wir mit `math.isclose`.)

## A3 · Kosinus-Ähnlichkeit

Wie ähnlich sind zwei Vektoren in ihrer **Richtung**, unabhängig von ihrer Länge? Das misst der Kosinus des Winkels zwischen ihnen:

$$\cos\theta = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}, \qquad \lVert a \rVert = \sqrt{a \cdot a}$$

Werte nahe 1 heißen „zeigt in dieselbe Richtung“, 0 heißt „senkrecht“, −1 heißt „entgegengesetzt“. In Woche 4 vergleicht ihr so die Bedeutung von Texten.

In NumPy: Skalarprodukt `a @ b`, Norm `np.linalg.norm(a)`.

### TODO 3 · Kosinus-Ähnlichkeit als Funktion

In [ ]:
def kosinus(a, b):
    # TODO 3: Formel oben mit a @ b und np.linalg.norm umsetzen
    pass

In [ ]:
v = np.array([3.0, 4.0])
check("TODO 3a gleicher Vektor → 1", lambda: math.isclose(kosinus(v, v), 1.0))
check("TODO 3b dreifache Länge → trotzdem 1", lambda: math.isclose(kosinus(v, 3 * v), 1.0))
check("TODO 3c senkrecht → 0", lambda: math.isclose(kosinus(v, np.array([-4.0, 3.0])), 0.0, abs_tol=1e-12))
check("TODO 3d entgegengesetzt → −1", lambda: math.isclose(kosinus(v, -v), -1.0))

## A4 · 10.000 Schwingungsspektren durchsuchen

**Szenario:** Die CNC-Drehmaschine M-03 läuft seit Tagen unrund (siehe Störungsmeldungen S-001, S-013, S-021). Die Instandhaltung hat ein **Schwingungsspektrum** gemessen: die Schwingungsenergie in 64 Frequenzbändern.
In der Datenbank liegen **10.000 historische Messungen** mit bekannter Diagnose: `OK`, `Unwucht`, `Fehlausrichtung` oder `Lagerschaden`.

**Frage:** Welche historischen Fälle sehen der aktuellen Messung am ähnlichsten, und welche Diagnose legt das nahe?

Die Daten sind synthetisch und werden in der nächsten Zelle erzeugt (fester Zufallsstartwert, alle bekommen dieselben Zahlen).

In [ ]:
rng = np.random.default_rng(42)
N_FAELLE, N_BAENDER = 10_000, 64
f = np.arange(N_BAENDER)
basis = np.exp(-f / 12)                                   # typischer Verlauf: tiefe Frequenzen dominieren
def peak(zentrum, breite):
    return np.exp(-((f - zentrum) ** 2) / (2 * breite ** 2))

MUSTER = {
    "OK":              basis,
    "Unwucht":         basis + 0.9 * peak(4, 1.0),                       # Spitze bei der Drehfrequenz
    "Fehlausrichtung": basis + 0.6 * peak(8, 1.0) + 0.4 * peak(16, 1.0), # Spitzen bei Vielfachen
    "Lagerschaden":    basis + 0.35 * peak(45, 3.0),                     # Energie bei hohen Frequenzen
}
KLASSEN = np.array(list(MUSTER))
diagnosen = rng.choice(KLASSEN, size=N_FAELLE, p=[0.88, 0.05, 0.05, 0.02])
amplitude = rng.lognormal(mean=0.0, sigma=0.35, size=N_FAELLE)          # Maschinen schwingen unterschiedlich stark
X = np.array([MUSTER[d] for d in diagnosen]) * amplitude[:, None] * (1 + 0.08 * rng.standard_normal((N_FAELLE, N_BAENDER)))
aktuell = 4.5 * MUSTER["Lagerschaden"] * (1 + 0.08 * rng.standard_normal(N_BAENDER))   # M-03 schwingt deutlich stärker

print("X:", X.shape, "| aktuelle Messung:", aktuell.shape)
print("Häufigkeit der Diagnosen:", dict(Counter(diagnosen)))

fig, ax = plt.subplots(figsize=(9, 3.2))
for name, m in MUSTER.items():
    ax.plot(f, m, label=name)
ax.plot(f, aktuell / 4.5, "k--", label="M-03 aktuell (skaliert)")
ax.set_xlabel("Frequenzband"); ax.set_ylabel("Energie"); ax.legend(); ax.set_title("Typische Spektren je Diagnose")
plt.show()

### TODO 4 · Alle 10.000 Ähnlichkeiten auf einmal
Statt 10.000-mal `kosinus(...)` aufzurufen, rechnet ihr mit der ganzen Matrix:
- `X @ aktuell` liefert alle 10.000 Skalarprodukte (Matrix × Vektor).
- `np.linalg.norm(X, axis=1)` liefert die Länge **jeder Zeile**.
- Die Division zweier Arrays gleicher Form geschieht elementweise.

Ergebnis: ein Array `aehnlichkeiten` mit 10.000 Werten.

In [ ]:
# TODO 4
aehnlichkeiten = None

print(aehnlichkeiten.shape if aehnlichkeiten is not None else None)

In [ ]:
check("TODO 4a Form", lambda: aehnlichkeiten.shape == (10_000,))
check("TODO 4b stimmt mit kosinus() überein", lambda: all(math.isclose(aehnlichkeiten[i], kosinus(X[i], aktuell)) for i in range(5)))

### TODO 5 · Die fünf ähnlichsten Fälle
`np.argsort(werte)` liefert die **Indizes**, die das Array aufsteigend sortieren. Für die größten Werte dreht ihr die Reihenfolge um (`[::-1]`) und nehmt die ersten fünf.
Bestimmt `top5` (Indizes) und `diagnose_vorschlag` (die häufigste Diagnose unter diesen fünf, zum Beispiel mit `Counter(...).most_common(1)`).

In [ ]:
# TODO 5
top5 = None
diagnose_vorschlag = None

if top5 is not None:
    for i in top5:
        print(f"Fall {i:5d}  Ähnlichkeit {aehnlichkeiten[i]:.4f}  Diagnose: {diagnosen[i]}")
print("Vorschlag:", diagnose_vorschlag)

In [ ]:
check("TODO 5a fünf Indizes", lambda: len(top5) == 5 and aehnlichkeiten[top5[0]] == aehnlichkeiten.max())
check("TODO 5b Diagnose", lambda: diagnose_vorschlag == "Lagerschaden")

> **Industriebezug:** Genau so arbeiten fallbasierte Diagnosesysteme. Und genau so funktioniert ab Woche 7 die Suche in einem RAG-System, nur dass die Vektoren dann Bedeutungen von Texten sind statt Frequenzbänder.

---
# Teil B · Web-APIs und JSON

## B1 · Ein Request an eine Wetter-API

Eine **API** ist eine Schnittstelle, über die ein Programm einen Dienst nutzt. Bei Web-APIs schickt ihr einen **HTTP-Request** an eine URL und bekommt eine **Response** mit Statuscode und meist **JSON** zurück.
Wir nutzen [Open-Meteo](https://open-meteo.com/en/docs): frei für nicht-kommerzielle Nutzung, ohne Key. Parameter übergebt ihr als Dictionary, `requests` baut daraus die URL.

In [ ]:
URL = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 49.01, "longitude": 8.40, "current": "temperature_2m"}   # Karlsruhe

antwort = requests.get(URL, params=params, timeout=10)
print("URL:", antwort.url)
print("Statuscode:", antwort.status_code)
daten = antwort.json()                       # JSON-Text → Python-Dictionary
print(json.dumps(daten, indent=2, ensure_ascii=False)[:600])

JSON wird in Python zu den Strukturen aus Woche 1: Objekte `{}` werden zu **Dictionaries**, Arrays `[]` zu **Listen**.

## B2 · Heizbedarf der Werkshalle

Die Halle der Nordwerk GmbH wird beheizt. Ein einfaches Maß für den Heizbedarf ist die **Heizgradzahl** (vereinfacht, Heizgrenze 15 °C):

$$\text{HGZ} = \sum_{\text{Tage}} \max(0,\; 15\,°\text{C} - T_{\text{Tag}})$$

### TODO 6 · Tagesmittel holen und Heizgradzahl berechnen
1. Ergänzt `params_tage`, sodass die API die **Tagesmitteltemperatur** (`daily`: `"temperature_2m_mean"`) der **letzten 7 Tage plus heute** liefert (`past_days`: 7, `forecast_days`: 1). Bei Tageswerten verlangt die API eine Zeitzone, nutzt `"Europe/Berlin"`.
2. Macht aus der Liste in der Antwort ein NumPy-Array `temps` und berechnet `hgz` ohne Schleife mit `np.maximum`.

Falls die API nicht erreichbar ist, nutzt die Zelle automatisch Beispieldaten, damit ihr weiterarbeiten könnt.

In [ ]:
BEISPIEL = {"daily": {"time": [f"2026-11-0{i}" for i in range(1, 9)],
                      "temperature_2m_mean": [9.8, 8.4, 7.1, 6.5, 8.9, 10.2, 7.7, 5.9]}}

# TODO 6a: Parameter ergänzen
params_tage = {
    "latitude": 49.01,
    "longitude": 8.40,
}

try:
    r = requests.get(URL, params=params_tage, timeout=10)
    r.raise_for_status()                      # wirft einen Fehler bei Statuscode 4xx/5xx
    tage = r.json()
except Exception as e:
    print("⚠️ API nicht erreichbar oder Parameter falsch, nutze Beispieldaten:", e)
    tage = BEISPIEL

# TODO 6b
temps = None
hgz = None

print("Tage:", tage.get("daily", {}).get("time"))
print("Temperaturen:", temps, "| Heizgradzahl:", hgz)

In [ ]:
check("TODO 6a Parameter", lambda: params_tage.get("daily") == "temperature_2m_mean" and params_tage.get("past_days") == 7 and "timezone" in params_tage)
check("TODO 6b temps ist ein Array mit einem Wert pro Tag", lambda: isinstance(temps, np.ndarray) and temps.ndim == 1 and len(temps) == len(tage["daily"]["time"]) >= 7)
check("TODO 6c Heizgradzahl", lambda: math.isclose(hgz, sum(max(0, 15 - t) for t in temps)))

> **Statuscodes:** 200 heißt OK. 400 heißt, eure Anfrage ist fehlerhaft (probiert `"temperatur_2m_mean"` mit Tippfehler). 401 heißt nicht autorisiert, 429 zu viele Anfragen, 5xx ein Fehler auf Serverseite.

---
# Teil C · Euer erster Request an ein Sprachmodell

## C1 · Schlüssel sicher laden

Wir nutzen die **KI-Toolbox des KIT**. Die Modelle mit `kit.` im Namen laufen auf Servern des SCC am KIT, eure Eingaben verlassen das KIT nicht. Die Toolbox bietet eine **OpenAI-kompatible Schnittstelle**: Wir nutzen die Bibliothek `openai` und geben nur eine andere Adresse (`base_url`) an.

Der Schlüssel kommt aus dem Colab-Secret, **nie** aus dem Code. Lauft ihr nicht in Colab, liest die Zelle stattdessen die Umgebungsvariable `KIT_TOOLBOX_KEY`.

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:                       # nicht in Colab
    import os
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
except Exception as e:                    # Secret fehlt oder Notebook-Zugriff nicht freigegeben
    API_KEY = None
    print("⚠️ Secret nicht lesbar:", type(e).__name__, "→ Woche 0, Kapitel 8")

API_KEY = API_KEY.strip() if API_KEY else None
BASE_URL = "https://ki-toolbox.scc.kit.edu/api/v1"   # Endpunkt der KI-Toolbox
client = OpenAI(api_key=API_KEY or "fehlt", base_url=BASE_URL, max_retries=3)
MODEL = "kit.mistral-small-4-119b-a8b"
print("Schlüssel geladen:", bool(API_KEY), "| Modell:", MODEL)   # niemals den Schlüssel selbst ausgeben!

Welche Modelle gibt es? Die Toolbox liefert die Liste selbst:

In [ ]:
modelle = sorted(m.id for m in client.models.list())
print("Am KIT gehostet:", [m for m in modelle if m.startswith("kit.")])

## C2 · Anatomie eines Calls

Ein Call an die **Chat-Completions-Schnittstelle** braucht drei Dinge: das Modell, eine Obergrenze für die Antwortlänge (`max_tokens`) und die Nachrichten. Rolle und Regeln gibt eine **System-Nachricht** am Anfang der Liste vor.

In [ ]:
antwort = client.chat.completions.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "system", "content": "Antworte auf Deutsch in einem Satz."},
              {"role": "user", "content": "Was ist eine Spindel bei einer Drehmaschine?"}],
)

print("Text:         ", antwort.choices[0].message.content)
print("finish_reason:", antwort.choices[0].finish_reason)   # stop = fertig; length = abgeschnitten
print("usage:        ", antwort.usage.prompt_tokens, "Input-Tokens,", antwort.usage.completion_tokens, "Output-Tokens")

### TODO 7 · Eine Störungsmeldung zusammenfassen lassen
Baut aus der Meldung unten einen `prompt` (f-String mit Maschine, Kategorie, Stillstand und Beschreibung) und schickt ihn mit dem gegebenen System-Prompt an das Modell.
Speichert die Antwort in `antwort7`.

In [ ]:
MELDUNGEN = [
    {"id": "S-004", "maschine": "M-05 Roboter-Schweißzelle", "kategorie": "elektrisch", "stillstand_min": 240,
     "beschreibung": "Schweißstromquelle ausgefallen, Tausch durch Instandhaltung"},
    {"id": "S-013", "maschine": "M-03 CNC-Drehmaschine", "kategorie": "mechanisch", "stillstand_min": 90,
     "beschreibung": "Spindel erneut unrund, Lagerwechsel geplant"},
    {"id": "S-024", "maschine": "M-06 Montagelinie A", "kategorie": "elektrisch", "stillstand_min": 75,
     "beschreibung": "Schrauber 2 liefert falsches Drehmoment"},
]
SYSTEM = ("Du bist Assistent der Instandhaltung in einem Fertigungsbetrieb. "
          "Fasse Störungsmeldungen für den Schichtbericht in höchstens zwei sachlichen Sätzen zusammen.")

meldung = MELDUNGEN[1]

# TODO 7: prompt bauen und das Modell aufrufen (model, max_tokens=150, messages mit system- und user-Nachricht)
prompt = None
antwort7 = None

if antwort7 is not None:
    print(antwort7.choices[0].message.content)

In [ ]:
check("TODO 7a Prompt enthält die Beschreibung", lambda: meldung["beschreibung"] in prompt)
check("TODO 7b Antwort erhalten", lambda: len(antwort7.choices[0].message.content) > 0 and antwort7.usage.completion_tokens > 0)

## C3 · Eine Funktion für alle LLM-Aufrufe

Ab jetzt laufen alle Aufrufe über eine kleine Funktion `llm(...)`. Das hat einen praktischen Grund: Wollt ihr später einen anderen Anbieter oder ein lokales Modell nutzen, ändert ihr **nur diese Funktion**.

> **Anbieterwechsel:** Die Schnittstelle von OpenAI ist ein verbreiteter Standard. Viele Anbieter und lokale Werkzeuge wie [Ollama](https://ollama.com) bieten sie ebenfalls an. Für einen Wechsel ändert ihr meist nur `base_url`, den Schlüssel und den Modellnamen. Genau das machen wir mit der KI-Toolbox.

In [ ]:
def llm(prompt, system=None, max_tokens=300, model=MODEL):
    """Schickt einen Prompt an das Sprachmodell. Gibt (Antworttext, usage) zurück."""
    nachrichten = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    antwort = client.chat.completions.create(model=model, max_tokens=max_tokens, messages=nachrichten)
    return antwort.choices[0].message.content, antwort.usage

## C4 · Was würde das kosten?

In der KI-Toolbox entstehen euch keine persönlichen Kosten, das KIT trägt den Betrieb. Es gilt **Fair Use**: üblicher Umfang ist in Ordnung, große automatisierte Läufe werden vorher mit dem SCC abgesprochen. Tokens sind trotzdem eine knappe Ressource: Sie bestimmen Rechenzeit, Wartezeit und wie viel in den Kontext passt.

Kommerzielle Anbieter rechnen pro Token ab, getrennt nach Input und Output. Als **Referenz** nehmen wir ein günstiges kommerzielles Modell mit **1 US-Dollar pro Million Input-Tokens** und **5 US-Dollar pro Million Output-Tokens** (so etwa Claude Haiku 4.5 laut Anthropic, Stand 04.10.2026):

$$K_{\text{Cent}} = \frac{n_{\text{in}} \cdot p_{\text{in}} + n_{\text{out}} \cdot p_{\text{out}}}{10^6} \cdot 100$$

### TODO 8 · Kosten berechnen und aufsummieren
1. Schreibt `kosten_cent(usage)`, die zurückgibt, was ein Call zum Referenzpreis in US-Cent kosten würde.
2. Lasst alle drei `MELDUNGEN` mit `llm(...)` zusammenfassen und summiert die Kosten in `gesamt_cent` (Akkumulator!). Die Zusammenfassungen sammelt ihr in der Liste `zusammenfassungen`.

In [ ]:
PREIS_INPUT_USD = 1.00    # Referenzpreis pro 1 Mio. Input-Tokens (kommerziell)
PREIS_OUTPUT_USD = 5.00   # pro 1 Mio. Output-Tokens

def kosten_cent(usage):
    # TODO 8a
    pass

zusammenfassungen = []
gesamt_cent = 0.0
for m in MELDUNGEN:
    # TODO 8b: Prompt bauen (wie in TODO 7), llm(...) aufrufen, Text anhängen, Kosten addieren
    pass

for z in zusammenfassungen:
    print("•", z)
print(f"Kommerziell wären das: {gesamt_cent:.4f} US-Cent")

In [ ]:
check("TODO 8a 1 Mio. Input-Tokens → 100 Cent", lambda: math.isclose(kosten_cent(SimpleNamespace(prompt_tokens=1_000_000, completion_tokens=0)), 100))
check("TODO 8a 1 Mio. Output-Tokens → 500 Cent", lambda: math.isclose(kosten_cent(SimpleNamespace(prompt_tokens=0, completion_tokens=1_000_000)), 500))
check("TODO 8b drei Zusammenfassungen, Kosten > 0", lambda: len(zusammenfassungen) == 3 and gesamt_cent > 0)

> **Zum Nachdenken:** Wie viele solcher Zusammenfassungen bekäme man kommerziell für einen US-Dollar? Und warum lohnt es sich auch ohne Kosten, Antworten kurz zu halten? (Wartezeit, geteilte Rechenleistung am KIT, Kontextgrenzen)

---
## Bonus

**B1 · Kosinus oder euklidischer Abstand?** Sucht die fünf nächsten Fälle mit dem **euklidischen Abstand** $\lVert x_i - a \rVert$ (`np.linalg.norm(X - aktuell, axis=1)`, aufsteigend sortiert). Welche Diagnose kommt jetzt heraus, und warum?
Normiert danach alle Vektoren auf Länge 1 und wiederholt die Suche. Was fällt auf? (Für Einheitsvektoren gilt $\lVert a - b \rVert^2 = 2 - 2\cos\theta$.)

**B2 · Das SDK ist nur HTTP.** Schickt denselben Request wie in C2 direkt mit `requests.post` an `BASE_URL + "/chat/completions"` (Header `Authorization: Bearer <Schlüssel>`) und vergleicht die JSON-Antwort mit dem SDK-Objekt.

In [ ]:
# B1

In [ ]:
# B2

## Zusammenfassung

- **NumPy** rechnet elementweise auf ganzen Arrays. Das ist kürzer und um ein Vielfaches schneller als Schleifen.
- **Masken** filtern ohne Schleife, **Broadcasting** erweitert Skalare und Vektoren automatisch auf passende Formen.
- Die **Kosinus-Ähnlichkeit** vergleicht Richtungen und ignoriert die Länge. Matrix × Vektor liefert tausende Vergleiche in einem Schritt.
- **Web-APIs** sprechen HTTP und JSON. JSON wird zu Dictionaries und Listen. Statuscodes sagen, ob es geklappt hat.
- **API-Schlüssel** gehören in Secrets. Ein LLM-Call besteht aus Modell, `max_tokens` und Nachrichten, die System-Nachricht steht vorne. Tokens bestimmen Wartezeit, Kontextgrenzen und bei kommerziellen Anbietern die Kosten.

### Ausblick auf Woche 3
Das Modell hat geantwortet. Aber **wie hat es das gelernt?** Nächste Woche schauen wir unter die Haube: Was heißt „Lernen“ für eine Maschine? Mit Loss-Funktionen, Gradient Descent und eurem ersten selbst trainierten Modell.